# D1.1 · From alert queue to loop operator

**Function D — Security Operations → The SOC Analyst & Detection Engineer**  ·  *AI for Security*

Builds on **[C2.9 · Research as institutional capital](https://spbreed.github.io/cyber-commons/lessons/C2.9.html)**.

| | |
|---|---|
| Open-source tooling | Wazuh, OpenSearch |
| Open-weight models | GLM-4.6 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


The classic SOC job is a queue: alerts arrive, an analyst reads each one,
decides, and moves on. The constraint is human attention, and it does not scale
— which is why tier-1 burnout and alert fatigue are structural rather than
cultural problems.

The agentic version replaces "read every alert" with "operate a loop that reads
every alert". The analyst's job becomes:

- deciding **what the loop is allowed to conclude** (the verifier, B2.2),
- deciding **what it may do about it** (the tool policy, A3.5),
- and handling the cases it escalates.

The skill that transfers is not triage speed. It is knowing which signals the
loop may believe — because a triage loop with a weak verifier closes true
positives at machine speed, and closing a true positive is silent.

> **About the model in this notebook.** It runs offline against a deterministic
> replay so the lesson executes on a Kaggle kernel with no network. To run the
> same triage against a real open-weight model:
>
> ```bash
> ollama pull glm-4.6            # or kimi-k2, llama3.3
> export OPENAI_BASE_URL=http://localhost:11434/v1 OPENAI_API_KEY=ollama MODEL=glm-4.6
> ```

## 2 · Demo — the queue, and the loop that reads it

In [ ]:
import time
from dataclasses import dataclass, field

@dataclass
class Alert:
    aid: str; rule: str; actor: str; target: str; severity: str
    truth: str          # held out from the loop: "tp" or "fp"

QUEUE = [
 Alert("A-01","impossible travel","dana@corp","vpn-eu","medium","fp"),
 Alert("A-02","metadata service access","patch-agent","169.254.169.254","critical","tp"),
 Alert("A-03","failed logins x40","svc-etl","auth","medium","fp"),
 Alert("A-04","secret path read","patch-agent","/home/app/.aws/credentials","high","tp"),
 Alert("A-05","new admin group member","sam@corp","group:admins","high","tp"),
 Alert("A-06","port scan detected","scanner-01","10.0.0.0/24","low","fp"),
 Alert("A-07","egress to unlisted host","triage-agent","collect.example.com","high","tp"),
 Alert("A-08","expired certificate","www","tls","low","fp"),
]
print(f"queue: {len(QUEUE)} alerts, "
      f"{sum(a.truth=='tp' for a in QUEUE)} true positives")
for a in QUEUE:
    print(f"   {a.aid} {a.severity:8s} {a.rule:24s} {a.actor}")

In [ ]:
class ReplayTriage:
    """DETERMINISTIC REPLAY — not a language model. Stands in for a triage model."""
    VERDICTS = {
     "A-01": ("close", 0.88, "corporate VPN egress in Frankfurt; matches this user's pattern"),
     "A-02": ("escalate", 0.97, "link-local metadata endpoint from a non-human identity"),
     "A-03": ("close", 0.71, "service account retry storm after a credential rotation"),
     "A-04": ("escalate", 0.93, "agent read a cloud credential path outside its workspace"),
     "A-05": ("escalate", 0.64, "privileged group change; needs the change ticket checked"),
     "A-06": ("close", 0.90, "authorised internal scanner, scheduled window"),
     "A-07": ("escalate", 0.95, "egress to a host not on the allowlist"),
     "A-08": ("close", 0.99, "hygiene finding, not a security event"),
    }
    def triage(self, alert):
        verdict, conf, why = self.VERDICTS[alert.aid]
        return {"aid": alert.aid, "verdict": verdict, "confidence": conf, "why": why}

model = ReplayTriage()
results = [model.triage(a) for a in QUEUE]
truth = {a.aid: a.truth for a in QUEUE}

print(f"{'alert':7s}{'verdict':10s}{'conf':>6}{'truth':>7}  reasoning")
print("-" * 92)
for r in results:
    t = truth[r["aid"]]
    correct = (r["verdict"] == "escalate") == (t == "tp")
    flag = "" if correct else "   ← WRONG"
    print(f"{r['aid']:7s}{r['verdict']:10s}{r['confidence']:>6.2f}{t:>7}{flag}  {r['why'][:44]}")

## 3 · Where it breaks — closing a true positive is silent

Every triage decision has two error directions and they are not symmetric. Escalating a false positive costs an analyst ten minutes. **Closing a true positive costs you the incident**, and nothing tells you it happened.

In [ ]:
def confusion(results, truth):
    tp = fp = tn = fn = 0
    missed = []
    for r in results:
        esc = r["verdict"] == "escalate"
        real = truth[r["aid"]] == "tp"
        if esc and real:      tp += 1
        elif esc and not real: fp += 1
        elif not esc and real: fn += 1; missed.append(r["aid"])
        else:                  tn += 1
    return {"escalated_correctly": tp, "false_escalations": fp,
            "closed_correctly": tn, "CLOSED_TRUE_POSITIVES": fn,
            "missed": missed,
            "analyst_minutes_saved": tn * 10,
            "incidents_missed": fn}

c = confusion(results, truth)
for k, v in c.items(): print(f"{k:26s}{v}")

print("\nNow lower the escalation bar and watch the trade:")
for threshold in (0.5, 0.7, 0.9, 0.99):
    esc = [r for r in results if r["verdict"] == "escalate" or r["confidence"] < threshold]
    adj = [{**r, "verdict": "escalate" if (r["verdict"] == "escalate" or
            r["confidence"] < threshold) else "close"} for r in results]
    cc = confusion(adj, truth)
    print(f"   close only above conf {threshold:.2f} → "
          f"missed {cc['incidents_missed']}, analyst minutes saved "
          f"{cc['analyst_minutes_saved']}")

## 4 · The control — the loop may close, but not silently

Three rules make an agentic triage loop safe to run, and none of them is about model quality.

In [ ]:
RULES = {
 "1. never close above a severity threshold":
   "critical and high alerts may be enriched and ranked, never auto-closed",
 "2. sample the closures":
   "a fixed fraction of auto-closed alerts go to a human, always",
 "3. measure closures against ground truth":
   "when an incident is found later, check whether the loop closed a related alert",
}
for k, v in RULES.items(): print(f"{k}\n     {v}")

def safe_triage(alert, verdict, confidence, sample_rate=0.1, seed=0):
    import random, zlib
    # NOT hash(): Python randomises str hashing per process (PYTHONHASHSEED),
    # so hash(alert.aid) picks a different sample on every run and on every
    # machine. crc32 is stable, which is what a sampling rule needs.
    rng = random.Random(zlib.crc32(alert.aid.encode()) % 1000 + seed)
    if verdict == "close" and alert.severity in ("critical", "high"):
        return "escalate", "rule 1: severity floor — never auto-close high/critical"
    if verdict == "close" and rng.random() < sample_rate:
        return "sample", "rule 2: routine closure sample for quality measurement"
    return verdict, ""

print()
adjusted = []
for a in QUEUE:
    r = model.triage(a)
    v, why = safe_triage(a, r["verdict"], r["confidence"])
    adjusted.append({**r, "verdict": "escalate" if v == "escalate" else
                     ("close" if v == "close" else "close")})
    print(f"   {a.aid} {a.severity:8s} {r['verdict']:9s} → {v:9s} {why}")

c2 = confusion(adjusted, truth)
print(f"\nbefore: missed {c['incidents_missed']}   after: missed {c2['incidents_missed']}")
assert c2["incidents_missed"] <= c["incidents_missed"]

## What you just proved

The triage loop escalates 4 alerts and closes 4, matching ground truth on all 8. Lowering the confidence bar trades analyst minutes against missed incidents. The severity floor converts any high or critical closure into an escalation, and the closure sampling routes a fraction of routine closures to a human for quality measurement.

## Your turn

Ask your SOC one question: when an incident is confirmed, does anyone check whether an earlier alert about it was closed? If nobody does, you have no measurement of your false-negative rate — with or without an agent.

---

**Next → [D1.2 · Context that makes triage work](https://spbreed.github.io/cyber-commons/lessons/D1.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*